# Kubernetes GitOps with Gitea {background-color="black" background-image="https://images.unsplash.com/photo-1618401471353-b98afee0b2eb?w=1600" background-size="cover" background-opacity="0.4"}

# The Course Capstone {background-color="#1a1a2e" background-image="https://images.unsplash.com/photo-1558494949-ef010cbdcc31?w=1600" background-size="cover" background-opacity="0.3"}

## Everything Converges Here

```{mermaid}
timeline
    title Cloud Systems Lab 2026
    Lesson 1-2 : Docker fundamentals
             : Compose
    Lesson 3-4 : Docker Swarm
              : Gitea CI/CD for Swarm
    Lesson 5-6 : IaC with OpenTofu
              : Multipass VMs
              : AWS CloudFormation
    Lesson 7   : Gitea + act_runner (shell)
              : Automated Swarm provisioning
    Lesson 8   : GitHub Actions + AWS OIDC
    Lesson 9   : Kubernetes with kubeadm
              : Multipass cluster
    Lesson 10  : GitOps for K8s
              : infra + app pipelines
```

# Recap {background-color="white" background-image="https://images.unsplash.com/photo-1497633762265-9d179a990aa6?q=80&w=2073&" background-size="cover" background-opacity="0.7"}

## What We Built in Last Lesson

::::{.columns}

::: {.fragment .column width="50%"}

**The cluster**

```bash
# 1 — Provision VMs
tofu apply

# 2 — Configure all nodes
ansible-playbook -i terraform/hosts.ini \
  ansible/00-prerequisites.yml

# 3 — Bootstrap control plane
ansible-playbook -i terraform/hosts.ini \
  ansible/01-control-plane.yml

# 4 — Join workers
ansible-playbook -i terraform/hosts.ini \
  ansible/02-workers.yml
```

:::

::: {.fragment .column width="50%"}

**Result after ~10 minutes**

```
NAME              STATUS   ROLES           AGE
control-plane-1   Ready    control-plane   8m
worker-1          Ready    <none>          5m
worker-2          Ready    <none>          5m
```

We ran all 4 commands manually from the terminal.

::: {.callout-warning}
**Same old problem:** not reproducible, requires your laptop, no audit trail, credentials on disk.
:::

:::

::::

## The Right Tool for Each Layer

::::{.columns}

::: {.fragment .column width="40%"}

```{mermaid}
flowchart TD
    INFRA["🏗️ Infrastructure Layer<br/>Multipass VMs · SSH keys<br/>Ansible inventory"]:::infra
    OS["🖥️ OS / Node Layer<br/>swap · kernel modules<br/>containerd · sysctl"]:::os
    K8S["☸️ Kubernetes Layer<br/>kubeadm init · kubeadm join<br/>etcd · kube-apiserver<br/>kubelet · Flannel CNI"]:::k8s
    APP["📦 Workload Layer<br/>Deployments · Services<br/>ConfigMaps · Scaling<br/>Rolling updates"]:::app

    INFRA -->|"VMs running<br/>inventory written"| OS
    OS -->|"nodes prepared"| K8S
    K8S -->|"kubectl get nodes ✅"| APP

    classDef infra fill:#7B68EE,color:#fff,stroke:#4b3fa0
    classDef os   fill:#E74C3C,color:#fff,stroke:#a93226
    classDef k8s  fill:#326CE5,color:#fff,stroke:#1a4a9f
    classDef app  fill:#27AE60,color:#fff,stroke:#1a7a42
```

:::

::: {.fragment .column width="60%"}

| Layer | Tool | Responsibility |
|---|---|---|
| 🏗️ Infrastructure | **OpenTofu** | VMs, SSH keys, Ansible inventory |
| 🖥️ OS / Node | **Ansible** | swap, modules, containerd, kubelet |
| ☸️ Kubernetes | **kubeadm** (via Ansible) | cluster bootstrap, TLS, etcd, CNI |
| 📦 Workloads | **kubectl** | Deployments, Services, rollouts |

::: {.fragment}
*"OpenTofu gives Ansible machines to configure. Ansible gives kubectl a cluster to deploy into."*
:::

::: {.callout-important .fragment}
**Why not `ansible.kubernetes.core.k8s`?**  
It wraps `kubectl apply` — you'd be duplicating Kubernetes's own reconciliation engine. The K8s API *is* the operator. Ansible adds no value past the cluster boundary.
:::

:::

::::


# Pipeline Design {background-color="#2d1b69" background-image="https://images.unsplash.com/photo-1556761175-4b46a572b786?w=1600" background-size="cover" background-opacity="0.3"}

## One Repo fits All

::: {.callout-note title="All in One"}
The repo contains *both* cluster infrastructure (`terraform/`, `ansible/`) *and* application manifests (`k8s/`). In production you'd split these into two repos — one for infra, one per app.
:::

```{mermaid}
flowchart LR
    TF["📁 terraform/**"]:::tofu
    AN["📁 ansible/**"]:::ansible
    K8S["📁 k8s/**"]:::kubectl

    P1["provision<br/>tofu apply"]:::tofu
    P2["configure<br/>00/01/02-*.yml"]:::ansible
    P3["save kubeconfig<br/>~/k8s-config"]:::shell

    D1["kubectl apply -f k8s/"]:::kubectl
    D2["kubectl rollout status"]:::kubectl

    TF & AN -->|"push triggers infra.yml"| P1
    P1 --> P2 --> P3

    K8S -->|"push triggers deploy.yml"| D1
    D1 --> D2

    P3 -.->|"~/k8s-config<br/>persists on host"| D1

    classDef tofu    fill:#7B68EE,color:#fff,stroke:#4b3fa0
    classDef ansible fill:#E74C3C,color:#fff,stroke:#a93226
    classDef shell   fill:#555,color:#fff,stroke:#333
    classDef kubectl fill:#326CE5,color:#fff,stroke:#1a4a9f
```


## Repository Structure

::::{.columns}

::: {.column width="50%"}

```
k8s-gitea-cicd/
├── .gitea/
│   └── workflows/
│       ├── infra.yml      ← cluster lifecycle
│       ├── deploy.yml     ← workload deploy
│       └── destroy.yml    ← manual teardown
├── terraform/             ← same as Lesson 9
│   ├── main.tf            (state → ~/k8s-terraform.tfstate)
│   ├── variables.tf
│   └── inventory.tpl      (inventory → ~/k8s-hosts.ini)
├── ansible/               ← same playbooks
│   ├── 00-prerequisites.yml
│   ├── 01-control-plane.yml
│   └── 02-workers.yml
├── k8s/
│   ├── nginx-deployment.yaml
│   └── nginx-service.yaml
└── setup-repo.sh
```

:::

::: {.fragment .column width="50%"}

**Why separate triggers?**

```yaml
# infra.yml — only runs when cluster code changes
on:
  push:
    paths: ['terraform/**', 'ansible/**']

# deploy.yml — only runs when app manifests change
on:
  push:
    paths: ['k8s/**']
```

::: {.callout-tip}
Changing your `nginx-deployment.yaml` does **not** re-provision the cluster. Changing `variables.tf` does **not** redeploy your app.
:::

:::

::::

## Prerequisites — Gitea & act_runner

::::{.columns}

::: {.fragment .column width="50%"}

**Start Gitea** (`lab/gitea-setup/`)

```bash
cd lab/gitea-setup

# First time only — downloads binary + opens wizard
./install-gitea.sh
# → http://localhost:3000  (complete setup wizard)

# Subsequent runs — just start the server
./gitea web --config conf/app.ini --port 3000 &
```

::: {.callout-note .fragment}
Gitea stores all data under `lab/gitea-setup/data/` — no Docker, no system install.
:::

:::

::: {.fragment .column width="50%"}

**Register & start act_runner** (`lab/gitea-setup/`)

```bash
cd lab/gitea-setup

# First time only — downloads binary + registers runner
# Get a runner token at:
# http://localhost:3000/-/admin/runners → Create Runner
GITEA_TOKEN=<runner-token> ./register-runner.sh
# labels: self-hosted,linux,multipass

# Start the runner (keep this terminal open)
./act_runner daemon
```

::: {.callout-warning .fragment}
The runner **must be running** before you push — otherwise pipeline jobs queue indefinitely. Verify at `http://localhost:3000/-/admin/runners` (status: Online).
:::

:::

::::

::: {.fragment}

**✅ Do it now — bootstrap the repo and start the pipeline**

```bash
cd lab/k8s-gitea-cicd

# Create Gitea repo + enable Actions + upload secrets
GITEA_TOKEN=<your-pat> ./setup-repo.sh

# Copy to a clean dir OUTSIDE this repo, then push
rsync -a --exclude='.git' . ~/k8s-infra/
cd ~/k8s-infra
git init && git branch -M main
git remote add gitea http://localhost:3000/<user>/k8s-infra.git
git add . && git commit -m "feat: initial k8s cluster setup"
git push gitea main
# → infra.yml starts now. We'll verify the cluster in ~10 min.
```

:::


# State & Secrets {background-color="#1a2a1a" background-image="https://images.unsplash.com/photo-1614064641938-3bbee52942c7?w=1600" background-size="cover" background-opacity="0.3"}

## Host Runner — Persistent State


Act runner executes jobs directly on your laptop:

| File | Written by | Used by |
|---|---|---|
| `~/k8s-terraform.tfstate` | `tofu apply` | `tofu plan`, `tofu destroy` | 
| `~/k8s-hosts.ini` | `local_file.inventory` | all 3 Ansible playbooks | | 
| `/tmp/k8s_join_command.sh` | `01-control-plane.yml` | `02-workers.yml` | 
| `~/k8s-config` | infra configure job | deploy.yml | 





## Secrets Setup

Only two secrets are needed — the same pattern as Lesson 7:

| Secret | Value | Used in |
|---|---|---|
| `SSH_PRIVATE_KEY` | Content of `id_ed25519` | All pipeline jobs |
| `SSH_PUBLIC_KEY` | Content of `id_ed25519.pub` | `provision` (cloud-init) |

```bash
# Generate a fresh key pair for this lab
ssh-keygen -t ed25519 -f terraform/id_ed25519 -N ""

# Create repo + upload secrets (calls Gitea API)
GITEA_TOKEN=<your-pat> ./setup-repo.sh
```

::: {.fragment}

**How secrets are used in the pipeline:**

```yaml
- name: Write SSH key pair
  run: |
    echo "${{ secrets.SSH_PRIVATE_KEY }}" > terraform/id_ed25519
    echo "${{ secrets.SSH_PUBLIC_KEY }}"  > terraform/id_ed25519.pub
    chmod 600 terraform/id_ed25519
```

:::

::: {.callout-tip .fragment}
No KUBECONFIG secret needed — `~/k8s-config` is written directly on the host by the infra pipeline and reused by deploy.
:::

# `infra.yml` — Cluster Lifecycle {background-color="#4b3fa0" background-image="https://images.unsplash.com/photo-1451187580459-43490279c0fa?w=1600" background-size="cover" background-opacity="0.3"}

## Job 1: `provision` — OpenTofu

```yaml
jobs:
  provision:
    runs-on: self-hosted
    steps:
      - uses: actions/checkout@v4

      - name: Write SSH key pair
        run: |
          echo "${{ secrets.SSH_PRIVATE_KEY }}" > terraform/id_ed25519
          echo "${{ secrets.SSH_PUBLIC_KEY }}"  > terraform/id_ed25519.pub
          chmod 600 terraform/id_ed25519

      - name: tofu init
        run: tofu -chdir=terraform init

      - name: tofu plan
        run: tofu -chdir=terraform plan -out=tfplan

      - name: tofu apply
        run: tofu -chdir=terraform apply -auto-approve tfplan
```

::: {.callout-note .fragment}
State is at `~/k8s-terraform.tfstate` — `tofu plan` will show **no changes** if the cluster already exists (idempotent).
:::

## Job 2: `configure` — Ansible + kubeconfig

```yaml
  configure:
    runs-on: self-hosted
    needs: provision          # waits for VMs to exist
    steps:
      - uses: actions/checkout@v4

      - name: Write SSH private key
        run: |
          echo "${{ secrets.SSH_PRIVATE_KEY }}" > terraform/id_ed25519
          chmod 600 terraform/id_ed25519

      - name: Wait for VMs to be SSH-ready
        run: sleep 30

      - name: Install K8s prerequisites on all nodes
        run: ansible-playbook -i ~/k8s-hosts.ini ansible/00-prerequisites.yml

      - name: Bootstrap control plane
        run: ansible-playbook -i ~/k8s-hosts.ini ansible/01-control-plane.yml

      - name: Join worker nodes
        run: ansible-playbook -i ~/k8s-hosts.ini ansible/02-workers.yml

      - name: Save kubeconfig to host runner
        run: |
          CONTROL_PLANE_IP=$(grep ansible_host ~/k8s-hosts.ini \
            | head -1 | grep -oP 'ansible_host=\K\S+')
          scp -i terraform/id_ed25519 -o StrictHostKeyChecking=no \
            ubuntu@${CONTROL_PLANE_IP}:/home/ubuntu/.kube/config \
            ~/k8s-config
          sed -i "s|https://[0-9.]*:6443|https://${CONTROL_PLANE_IP}:6443|" \
            ~/k8s-config
          chmod 600 ~/k8s-config
          KUBECONFIG=~/k8s-config kubectl get nodes
```

::: {.fragment}

**✅ Do it now — verify the cluster is up**

```bash
# Pipeline should be complete by now — check nodes
KUBECONFIG=~/k8s-config kubectl get nodes
# NAME              STATUS   ROLES           AGE
# control-plane-1   Ready    control-plane   ~8m
# worker-1          Ready    <none>          ~5m
# worker-2          Ready    <none>          ~5m

# deploy.yml already ran on the initial push — check the app
KUBECONFIG=~/k8s-config kubectl get pods -o wide
KUBECONFIG=~/k8s-config kubectl get svc nginx
```

:::


# `deploy.yml` — Workload Pipeline {background-color="#1a3a5c" background-image="https://images.unsplash.com/photo-1558494949-ef010cbdcc31?w=1600" background-size="cover" background-opacity="0.3"}

## `deploy.yml` — Apply K8s Manifests

::::{.columns}

::: {.column width="55%"}

```yaml
on:
  push:
    branches: [main]
    paths: ['k8s/**']
  workflow_dispatch:

jobs:
  deploy:
    runs-on: self-hosted
    steps:
      - uses: actions/checkout@v4

      - name: Verify cluster is reachable
        run: KUBECONFIG=~/k8s-config kubectl get nodes

      - name: Apply manifests
        run: KUBECONFIG=~/k8s-config kubectl apply -f k8s/

      - name: Wait for rollout
        run: |
          KUBECONFIG=~/k8s-config \
            kubectl rollout status deployment/nginx \
            --timeout=120s

      - name: Show running pods
        run: KUBECONFIG=~/k8s-config kubectl get pods -o wide
```

:::

::: {.fragment .column width="45%"}

**The manifests (`k8s/`):**

```yaml
# nginx-deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: nginx
spec:
  replicas: 3
  selector:
    matchLabels: {app: nginx}
  template:
    spec:
      containers:
        - name: nginx
          image: nginx:1.27-alpine
```

```yaml
# nginx-service.yaml
kind: Service
spec:
  type: NodePort
  ports:
    - port: 80
      nodePort: 30080
```

:::

::::

::: {.fragment}

**✅ Do it now — curl the running app**

```bash
WORKER_IP=$(KUBECONFIG=~/k8s-config kubectl get nodes \
  -l '!node-role.kubernetes.io/control-plane' \
  -o jsonpath='{.items[0].status.addresses[0].address}')
curl http://${WORKER_IP}:30080
# Expected: nginx welcome page
```

:::


## Rolling Update — GitOps Loop

```{mermaid}
%%{init: {'theme': 'dark'}}
sequenceDiagram
    participant Dev as Developer
    participant Git as Gitea
    participant Runner as act_runner (shell)
    participant K8s as Kubernetes

    Dev->>Git: git push (change image: nginx:1.27 → 1.28)
    Git->>Runner: trigger deploy.yml
    Runner->>K8s: kubectl apply -f k8s/
    K8s->>K8s: rolling update (new pod, old pod)
    K8s-->>Runner: rollout complete
    Runner-->>Git: job success
    Git-->>Dev: ✓ pipeline green

    Note over Dev,K8s: Rollback: git revert → push → kubectl apply
```

::: {.fragment}

::: {.callout-tip}
**Zero-downtime update** — Kubernetes replaces pods one at a time, ensuring at least `replicas - 1` pods remain available. No Ansible, no SSH, no downtime.
:::

:::

::: {.fragment}

**✅ Do it now — rolling update, rollback & scale**

```bash
cd ~/k8s-infra

# Update image → triggers deploy.yml
sed -i 's/nginx:1.27-alpine/nginx:1.28-alpine/' k8s/nginx-deployment.yaml
git add k8s/nginx-deployment.yaml
git commit -m "feat: upgrade nginx to 1.28"
git push gitea main

# Rollback via git revert
git revert HEAD --no-edit && git push gitea main

# Scale up
sed -i 's/replicas: 3/replicas: 5/' k8s/nginx-deployment.yaml
git add k8s/nginx-deployment.yaml
git commit -m "feat: scale nginx to 5 replicas"
git push gitea main
```

:::


# `destroy.yml` — Teardown {background-color="#3a0a0a" background-image="https://images.unsplash.com/photo-1555680202-c86f0e12f086?w=1600" background-size="cover" background-opacity="0.3"}

## Controlled Teardown

```yaml
name: Destroy K8s Cluster
on:
  workflow_dispatch:   # manual only — never automatic

jobs:
  destroy:
    runs-on: self-hosted
    steps:
      - uses: actions/checkout@v4

      - name: Write SSH key pair
        run: |
          echo "${{ secrets.SSH_PRIVATE_KEY }}" > terraform/id_ed25519
          echo "${{ secrets.SSH_PUBLIC_KEY }}"  > terraform/id_ed25519.pub
          chmod 600 terraform/id_ed25519

      - name: tofu destroy
        run: tofu -chdir=terraform destroy -auto-approve

      - name: Clean up host runner state
        run: |
          rm -f ~/k8s-config ~/k8s-hosts.ini
          rm -f /tmp/k8s_join_command.sh
```

::: {.fragment}

::: {.callout-important}
**Always run destroy at the end of the lab.** Multipass VMs keep consuming RAM even when idle. `~/k8s-terraform.tfstate` persists on the host — `tofu destroy` needs it to know which VMs to delete.
:::

:::

::: {.fragment}

**✅ Do it now — trigger destroy**

```bash
# In Gitea UI: k8s-infra → Actions → destroy.yml → Run workflow
# Or via API:
curl -s -X POST \
  http://localhost:3000/api/v1/repos/<user>/k8s-infra/actions/workflows/destroy.yml/dispatches \
  -H "Authorization: token <your-pat>" \
  -H "Content-Type: application/json" \
  -d '{"ref":"main"}'
```

:::


# Comparison & Course Summary {background-color="#1a1a2e" background-image="https://images.unsplash.com/photo-1451187580459-43490279c0fa?w=1600" background-size="cover" background-opacity="0.4"}

## Manual vs GitOps

| | Lesson 9 (manual) | Lesson 10 (GitOps) |
|---|---|---|
| **Reproducibility** | ❌ depends on your terminal | ✅ same result every push |
| **Audit trail** | ❌ no history | ✅ git log + pipeline history |
| **Credentials** | ❌ keys on your laptop | ✅ secrets in Gitea |
| **Rollback** | ❌ manual (`kubectl rollout undo`) | ✅ `git revert` → push |
| **Collaboration** | ❌ "works on my machine" | ✅ pull request + review |
| **Infra/app separation** | ❌ everything by hand | ✅ separate triggered pipelines |

::: {.fragment}

::: {.callout-tip}
**Next step beyond this course:** Argo CD or Flux CD — the same `kubectl apply` loop but running *inside* the cluster, continuously reconciling against a Git repository. The pipeline disappears; Git *is* the source of truth.
:::

:::

## Course Summary

::::{.columns}

::: {.column width="50%"}

**Tools mastered**

- **Docker** — containers, Compose, Swarm
- **OpenTofu** — declarative IaC, Multipass + AWS
- **Ansible** — agentless configuration management
- **Kubernetes** — pod scheduling, rolling updates, services
- **Gitea + act_runner** — self-hosted CI/CD
- **GitHub Actions + OIDC** — cloud-native pipelines

:::

::: {.fragment .column width="50%"}

**Patterns learned**

- Infrastructure as Code vs Configuration Management
- Right tool for the right layer (Ansible ≠ K8s app operator)
- Path-based pipeline triggers for separation of concerns
- State persistence strategies (local vs remote backend)
- Secrets management (Gitea, GitHub, OIDC)
- GitOps: git as single source of truth

:::

::::

::: {.fragment}

::: {.callout-note}
**What's next:** Argo CD / Flux CD for continuous GitOps inside K8s, Helm charts, multi-cluster federation, service mesh (Istio/Linkerd).
:::

:::